# Colab 16 · CNN-Bilddetektive
## Teil 1: Filter, Faltung und Pooling

**Zeit: 70 Minuten · Für Teams mit 2–3 Personen**

Wie kann eine KI später Bilder von **Pferden und Menschen** unterscheiden?
Dazu müssen wir zuerst verstehen, wie sie kleine Bildmuster verarbeitet.
Heute untersucht ihr diese Bausteine mit Zahlenbildern.

**Ihr müsst keinen Python-Code schreiben oder verändern.**
Führt die vorbereiteten Zellen mit **▶** aus. Der Code ist eingeklappt.
Bearbeitet die Zellen der Reihe nach. Die Beispielbilder entstehen direkt im Colab.
Eine normale CPU-Laufzeit genügt.

### So arbeitet ihr

**Vermuten → Ausführen → Beobachten → Erklären.**

Sprecht vor dem Ausführen kurz über eure Vermutung. Wechselt euch beim Bedienen ab.
In den interaktiven Aufgaben wählt ihr eine Antwort oder gebt Zahlen ein.
Danach klickt ihr auf **«Prüfen»**.
Führt die jeweilige Zelle zuerst mit **▶** aus, damit die Eingabefelder und Schaltflächen erscheinen.

**Eure schriftlichen Befunde und Erklärungen gehören in die Labordokumentation (Word).**
Die Auswahl- und Zahlenaufgaben löst ihr direkt im Colab.

| Abschnitt | Zeit |
|---|---:|
| 0 · Start und Lernziele | 5 Min. |
| 1 · Ein Bild besteht aus Zahlen | 5 Min. |
| 2 · Faltung Schritt für Schritt | 20 Min. |
| 3 · Verschiedene Filter, verschiedene Muster | 8 Min. |
| 4 · Max-Pooling: weniger Werte | 15 Min. |
| 5 · Die Bausteine im CNN | 8 Min. |
| 6 · Lerncheck ohne Hilfe | 9 Min. |
| **Total** | **70 Min.** |


### 0 · Das könnt ihr nach Teil 1 erklären und anwenden

Diese Lernziele gehören zum Stoff von **Colab 16 für die nächste Prüfung**.

- Ich kann ein kleines Graustufenbild als Zahlenraster lesen.
- Ich kann eine Filterantwort berechnen: **passende Werte multiplizieren, Produkte addieren**.
- Ich kann erklären, wie derselbe Filter über ein Bild wandert und eine **neue Merkmalskarte** erzeugt.
- Ich kann erklären, warum verschiedene Filter auf unterschiedliche Muster reagieren.
- Ich kann **Max-Pooling mit getrennten 2×2-Bereichen** durchführen und den Informationsverlust erklären.
- Ich kann Faltung, ReLU und Pooling unterscheiden. Ich weiss, welche Gewichte beim Training gelernt werden.

**Ausblick:** Die vollständige Unterscheidung von Pferden und Menschen folgt in Teil 2.
Heute berechnen wir noch keine solche Bildklasse.

**Startet jetzt die folgende Vorbereitungszelle.**


In [ ]:
#@title 0. Vorbereitung starten
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle
from matplotlib.colors import ListedColormap

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'savefig.facecolor': 'white', 'text.color': '#111827',
    'axes.labelcolor': '#111827', 'axes.edgecolor': '#64748b',
    'xtick.color': '#111827', 'ytick.color': '#111827',
    'font.family': 'DejaVu Sans', 'font.size': 12,
    'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'figure.dpi': 110, 'savefig.dpi': 140,
})

DARK = '#111827'
ORANGE = '#c45c0b'
BLUE = '#1d4ed8'
GREEN = '#15803d'
BLOCK_COLORS = ['#f3c35b', '#77bce8', '#8dd3b0', '#c4a5e8']
VERTICAL_FILTER = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]])
HORIZONTAL_FILTER = VERTICAL_FILTER.T.copy()
EDGE_IMAGE = np.tile([0, 0, 0, 1, 1, 1], (6, 1))


def apply_filter(image, kernel):
    height = image.shape[0] - kernel.shape[0] + 1
    width = image.shape[1] - kernel.shape[1] + 1
    result = np.zeros((height, width), dtype=int)
    for row in range(height):
        for column in range(width):
            patch = image[row:row + kernel.shape[0], column:column + kernel.shape[1]]
            result[row, column] = int(np.sum(patch * kernel))
    return result


def max_pool(image):
    height, width = image.shape
    if height % 2 or width % 2:
        raise ValueError('Dieses Beispiel benötigt gerade Seitenlängen.')
    return image.reshape(height // 2, 2, width // 2, 2).max(axis=(1, 3))


def draw_grid(ax, values, title, mode='response', low=None, high=None,
              visible=None, blocks=False, ring_maxima=False, highlight=None,
              font_size=15, coordinates=False):
    values = np.asarray(values)
    height, width = values.shape
    if blocks:
        colors = np.array([[BLOCK_COLORS[(row // 2) * 2 + column // 2]
                            for column in range(width)] for row in range(height)])
        if height == 2 and width == 2:
            colors = np.array(BLOCK_COLORS).reshape(2, 2)
        from matplotlib.colors import to_rgba
        rgba = np.array([[to_rgba(color) for color in line] for line in colors])
        ax.imshow(rgba, interpolation='nearest')
        color_map = None
        normalization = None
    else:
        if mode == 'image':
            color_map = plt.get_cmap('gray')
            low = 0 if low is None else low
            high = 1 if high is None else high
        else:
            color_map = plt.get_cmap('RdBu_r')
            limit = max(1, int(np.abs(values).max()))
            low = -limit if low is None else low
            high = limit if high is None else high
        from matplotlib.colors import Normalize
        normalization = Normalize(vmin=low, vmax=high)
        ax.imshow(values, cmap=color_map, vmin=low, vmax=high, interpolation='nearest')
    ax.set_title(title, pad=11, color=DARK)
    ax.set_xticks(np.arange(-0.5, width, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, height, 1), minor=True)
    ax.grid(which='minor', color='#64748b', linewidth=0.8)
    ax.tick_params(which='minor', bottom=False, left=False)
    if coordinates:
        ax.set_xticks(np.arange(width), labels=np.arange(1, width + 1))
        ax.set_yticks(np.arange(height), labels=np.arange(1, height + 1))
        ax.set_xlabel('Spalte', labelpad=5)
        ax.set_ylabel('Zeile', labelpad=5)
        ax.tick_params(length=0, labelsize=10)
    else:
        ax.set_xticks([])
        ax.set_yticks([])
    for row in range(height):
        for column in range(width):
            known = visible is None or bool(visible[row, column])
            label = str(int(values[row, column])) if known else '·'
            if not known:
                ax.add_patch(Rectangle((column - 0.5, row - 0.5), 1, 1,
                                       facecolor='#f1f5f9', edgecolor='#64748b',
                                       linewidth=0.8, zorder=2))
                text_color = '#64748b'
            elif blocks:
                text_color = DARK
            else:
                red, green, blue, _ = color_map(normalization(values[row, column]))
                luminance = 0.2126 * red + 0.7152 * green + 0.0722 * blue
                text_color = DARK if luminance > 0.53 else 'white'
            ax.text(column, row, label, ha='center', va='center',
                    color=text_color, fontsize=font_size, fontweight='bold', zorder=4)
    if blocks and height == 4 and width == 4:
        ax.axvline(1.5, color='white', linewidth=5)
        ax.axhline(1.5, color='white', linewidth=5)
    if ring_maxima:
        for block_row in range(2):
            for block_column in range(2):
                block = values[block_row * 2:block_row * 2 + 2,
                               block_column * 2:block_column * 2 + 2]
                local_row, local_column = np.unravel_index(np.argmax(block), block.shape)
                ax.add_patch(Circle((block_column * 2 + local_column,
                                     block_row * 2 + local_row), radius=0.37,
                                    fill=False, edgecolor=DARK, linewidth=2.3, zorder=5))
    if highlight is not None:
        row, column, size, color = highlight
        ax.add_patch(Rectangle((column - 0.5, row - 0.5), size, size,
                               fill=False, edgecolor=color, linewidth=3.4, zorder=6))
    ax.set_xlim(-0.53, width - 0.47)
    ax.set_ylim(height - 0.47, -0.53)
    ax.set_aspect('equal')


def finish_figure(fig, title, caption=None):
    fig.suptitle(title, fontsize=16, fontweight='bold', color=DARK)
    if caption:
        fig.text(0.5, 0.025, caption, ha='center', va='bottom',
                 fontsize=11, color=DARK)
    fig.tight_layout(rect=(0.01, 0.10 if caption else 0.02, 0.99, 0.90))
    plt.show()

print('Bereit. Führt die Experimente der Reihe nach aus.')
print('Ihr arbeitet mit Bildern und Zahlen. Ihr müsst den Code nicht öffnen.')


import io
from html import escape
import ipywidgets as widgets
from IPython.display import display

UI_STYLE = """
<style>
.cnn16-ui { background:#ffffff !important; color:#111827 !important;
  border:1px solid #cbd5e1; border-radius:10px; padding:14px; box-sizing:border-box; }
.cnn16-ui .widget-label, .cnn16-ui label, .cnn16-ui .widget-html-content,
.cnn16-ui .widget-checkbox, .cnn16-ui .widget-radio-box { color:#111827 !important; }
.cnn16-ui select, .cnn16-ui input { background:#ffffff !important;
  color:#111827 !important; border-color:#64748b !important; }
.cnn16-ui .widget-button { background:#dbeafe !important; color:#111827 !important;
  font-weight:600; border:1px solid #64748b; border-radius:5px; }
.cnn16-ui .widget-html-content { white-space:normal; line-height:1.45; }
.cnn16-ui .widget-radio-box label, .cnn16-ui .widget-checkbox label {
  white-space:normal !important; height:auto !important; min-height:25px; }
</style>
"""


def figure_png(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format='png', dpi=130, facecolor='white')
    plt.close(fig)
    return buffer.getvalue()


def panel(children):
    result = widgets.VBox([widgets.HTML(UI_STYLE), *children],
                          layout=widgets.Layout(width='100%', max_width='1100px'))
    result.add_class('cnn16-ui')
    return result


def feedback_html(message, kind='hint'):
    colors = {'correct': ('#dcfce7', '#14532d'),
              'wrong': ('#fff7ed', '#7c2d12'),
              'hint': ('#eff6ff', '#1e3a8a')}
    background, color = colors[kind]
    return (f'<div style="padding:9px 12px;border-radius:6px;'
            f'background:{background};color:{color} !important">{escape(message)}</div>')


## 1 · Ein Bild besteht aus Zahlen · 5 Minuten

Ein Graustufenbild besteht aus vielen kleinen Feldern: den **Pixeln**.
Jedes Pixel hat einen Helligkeitswert.

Für unsere ersten Bilder genügt diese vereinfachte Skala:

**0 = dunkel · 1 = hell**

Ein echtes Graustufenbild kann viele Helligkeitsstufen haben.
Wir verwenden hier nur zwei, damit ihr die Rechnungen gut verfolgen könnt.

**Vermutet zuerst:** Zeigen die beiden Raster dasselbe Bild?


In [ ]:
#@title 1. Ein Zahlenraster und sein Bild vergleichen
pixel_image = np.array([
    [0, 0, 0, 0, 0, 0],
    [0, 1, 1, 1, 1, 0],
    [0, 0, 1, 1, 0, 0],
    [0, 0, 1, 1, 0, 0],
    [0, 0, 1, 1, 0, 0],
    [0, 0, 0, 0, 0, 0],
])
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
draw_grid(axes[0], pixel_image, 'Zahlenraster: 6 × 6 = 36 Werte',
          mode='image', coordinates=True, font_size=14)
axes[1].imshow(pixel_image, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
axes[1].set_title('Dasselbe Raster als Bild', pad=11)
axes[1].set_xticks([])
axes[1].set_yticks([])
finish_figure(fig, 'Das Bild und die Zahlen gehören zusammen',
              '0 ist dunkel. 1 ist hell. Die Zahl bestimmt die Helligkeit eines Pixels.')


### Was bedeutet «das helle Muster nach rechts verschieben»?

Die hellen Felder werden **ein Feld weiter rechts** eingezeichnet.
Eine 1 bleibt eine 1. Sie steht danach an einer anderen Stelle.
Frei gewordene Felder werden wieder 0.
Das Bildraster selbst bleibt gleich gross.

Das nächste Beispiel zeigt eine einzelne Zeile. Das helle Muster bleibt vollständig im Bild.


In [ ]:
#@title 1b. Das helle Muster wandert ein Feld nach rechts
before_shift = np.array([[0, 0, 1, 1, 0, 0]])
after_shift = np.zeros_like(before_shift)
after_shift[:, 1:] = before_shift[:, :-1]
fig, axes = plt.subplots(1, 2, figsize=(10, 2.8))
draw_grid(axes[0], before_shift, 'Vorher: helle Felder an Stelle 3 und 4',
          mode='image', font_size=19)
draw_grid(axes[1], after_shift, 'Nachher: helle Felder an Stelle 4 und 5',
          mode='image', font_size=19)
finish_figure(fig, 'Die Zahlenwerte bleiben gleich. Ihre Positionen ändern sich.',
              'Die beiden 1 stehen jetzt ein Feld weiter rechts. Das freigewordene Feld wird 0.')


**Besprecht jetzt:** Wie viele Pixel hat das 6×6-Bild?
Was passiert mit den Zahlen, wenn das helle Muster ein Feld nach rechts verschoben wird?

### Wie nutzt der Computer die Nachbarschaft?

Der Computer betrachtet einen **kleinen Bildausschnitt mit benachbarten Pixeln**,
zum Beispiel ein 3×3-Feld mit neun Werten.
Diese neun Werte werden gemeinsam verarbeitet.

Ein Filter kann zum Beispiel die Werte links mit den Werten rechts vergleichen.
Sind links dunkle und rechts helle Felder, entsteht ein **Helligkeitsübergang**.
Ein anderer Filter kann oben mit unten vergleichen.
So können benachbarte Pixel zusammen einen Hinweis auf eine Kante liefern.


## 2 · Faltung Schritt für Schritt · 20 Minuten

### 2a · Zuerst nur ein Bildausschnitt

Ein **Filter** ist hier ein kleines Raster mit **3×3 Gewichten**.
Mit ihm berechnen wir für einen 3×3-Bildausschnitt einen neuen Wert.

1. Legt Bildausschnitt und Filter gedanklich übereinander.
2. Multipliziert die Werte an **derselben Position**.
3. Addiert die neun Produkte. Die Summe ist die **Filterantwort**.

Das Anwenden dieses Filters über das Bild nennen wir hier **Faltung**.
Für dieses einfache Rechenbeispiel ist der zusätzliche Bias-Wert 0.

**Vermutet:** Reagiert der folgende Filter auf den Übergang von dunkel zu hell?

**Merksatz für eure Vermutung:** Unser Filter hat links **−1**, in der Mitte **0**
und rechts **+1**. Er rechnet in jeder Zeile **rechter Wert minus linker Wert**.
Ist es rechts heller als links, wird die Antwort positiv.
Sind beide Seiten gleich hell, ergibt dieser Vergleich 0. Ist es links heller als rechts, wird die Antwort negativ.
Blickt zuerst auf Bildausschnitt und Filter. Überlegt dann, ob ihr eine positive, Null oder negative Antwort erwartet.


In [ ]:
#@title 2a. Eine Filterantwort vollständig nachvollziehen
first_patch = np.array([[0, 0, 1], [0, 0, 1], [0, 0, 1]])
products = first_patch * VERTICAL_FILTER
response = int(products.sum())
fig, axes = plt.subplots(1, 3, figsize=(11, 4.4))
draw_grid(axes[0], first_patch, '1 · Bildausschnitt', mode='image', font_size=19,
          highlight=(0, 0, 3, ORANGE))
draw_grid(axes[1], VERTICAL_FILTER, '2 · Filtergewichte', low=-1, high=1,
          font_size=19, highlight=(0, 0, 3, BLUE))
draw_grid(axes[2], products, '3 · Produkte an gleicher Stelle', low=-1, high=1,
          font_size=19, highlight=(0, 0, 3, GREEN))
finish_figure(fig, 'Passende Werte multiplizieren, dann alles addieren',
              f'Zeilensummen: 1 + 1 + 1 = {response}. Die Filterantwort ist {response}.')
print('Eine Zeile: 0 · (−1) + 0 · 0 + 1 · 1 = 1')
print('Alle drei Zeilen: 1 + 1 + 1 = 3')
print('Die 3 wird in eine neue Merkmalskarte eingetragen.')


### 2b · Jetzt wandert derselbe Filter über das Bild

Die Gewichte des Filters bleiben beim Verschieben **gleich**.
Nur der Bildausschnitt ändert sich. Jede Position liefert einen Wert für eine
**neue Merkmalskarte**, auch **Feature-Map** genannt.
Das Originalbild wird dabei nicht überschrieben.

Im nächsten Experiment seht ihr:

- **Orange:** den untersuchten 3×3-Ausschnitt im Bild.
- **Blau:** den unveränderten Filter.
- **Grün:** das aktuelle Ergebnis in der neuen Merkmalskarte.
- **Punkte:** Positionen, die noch nicht berechnet wurden.

**Zur Farbskala:** Im Bild bedeutet 0 dunkel und 1 hell.
In der Merkmalskarte stehen Filterantworten: Positive Werte sind rot,
negative Werte blau und 0 weiss. Die Farben helfen euch, die Zahlen zu vergleichen.

**Bedienung:** Stellt `position` nacheinander auf **1, 2, 3, 4 und 16**.
Klickt nach jeder Änderung auf **▶**. Die Positionen werden zeilenweise gezählt.

**Vor Position 2:** Berechnet den erwarteten Wert zuerst selbst.


In [ ]:
#@title 2b. Filter verschieben: Position auswählen und ▶ drücken
#@markdown **Position 1 bis 16:** derselbe Filter, ein anderer Bildausschnitt.
position = 16 #@param {type:"slider", min:1, max:16, step:1}

def draw_filter_position(selected_position):
    selected_position = int(selected_position)
    feature_map = apply_filter(EDGE_IMAGE, VERTICAL_FILTER)
    row, column = divmod(selected_position - 1, feature_map.shape[1])
    patch = EDGE_IMAGE[row:row + 3, column:column + 3]
    known = np.arange(feature_map.size).reshape(feature_map.shape) < selected_position
    fig, axes = plt.subplots(1, 3, figsize=(11.5, 4.8))
    draw_grid(axes[0], EDGE_IMAGE, 'Bild: 6 × 6 Werte', mode='image',
              highlight=(row, column, 3, ORANGE), font_size=13, coordinates=True)
    draw_grid(axes[1], VERTICAL_FILTER, 'Derselbe 3×3-Filter', low=-1, high=1,
              highlight=(0, 0, 3, BLUE), font_size=18)
    draw_grid(axes[2], feature_map, 'Neue Merkmalskarte: 4 × 4', low=-3, high=3,
              visible=known, highlight=(row, column, 1, GREEN),
              font_size=17, coordinates=True)
    answer = int(np.sum(patch * VERTICAL_FILTER))
    finish_figure(fig, f'Position {selected_position} von 16',
                  f'Ausschnitt beginnt in Zeile {row + 1}, Spalte {column + 1}. '
                  f'Summe der neun Produkte: {answer}.')
    terms = [f'{int(patch[line, 2])} − {int(patch[line, 0])}' for line in range(3)]
    print('Für diesen Filter genügt pro Zeile: rechter Wert minus linker Wert.')
    print('Rechnung: (' + ') + ('.join(terms) + f') = {answer}')
    print('Das Ergebnis steht in der grün markierten Zelle der neuen Karte.')

draw_filter_position(position)


**Untersucht die Ergebnisse:**

- Vergleicht Position 1 und 2. Warum entstehen unterschiedliche Werte?
- Welche Zahlen bleiben beim Verschieben gleich?
- Wo erscheinen die hohen Werte in der vollständigen Merkmalskarte?

**Warum ist die Merkmalskarte kleiner?**
Wir setzen den Filter nur dort auf, wo er **vollständig ins Bild passt**.
Wir verschieben ihn jeweils um ein Pixel. In unserem 6×6-Bild gibt es
dadurch 4×4 passende Positionen. In dieser Einheit fügen wir keinen Rand hinzu.

**Merksatz:** Eine Merkmalskarte enthält Filterantworten.
Ein hoher positiver Wert zeigt hier einen starken Übergang von dunkel links zu hell rechts.
Er bedeutet noch nicht «Pferd» oder «Mensch».


**Labordokumentation (Word) · Experiment 1:**
Haltet euren Kurzbefund zu Faltung und Merkmalskarte dort fest.


## 3 · Verschiedene Filter, verschiedene Muster · 8 Minuten

Ein Filter reagiert besonders auf **senkrechte Helligkeitsübergänge**.
Ein anderer reagiert auf **waagrechte Helligkeitsübergänge**.

### Filterdetektive: Welche Merkmalskarte passt?

Im Feld **«Beispiel»** stehen zehn Fälle zur Wahl:
fünf Bildmuster, jeweils mit einem Filter für senkrechte oder waagrechte Kanten.

1. Wählt ein Beispiel. Oben seht ihr das **Bild und den gesuchten Filter**.
2. Vergleicht darunter die drei vorgeschlagenen Merkmalskarten **A, B und C**.
3. Wählt die Karte, die **dieser Filter für dieses Bild** liefert.
4. Klickt auf **«Prüfen»**. Bei einem Fehler könnt ihr erneut überlegen und wählen.

**Pflicht:** Beispiele **1, 4, 5 und 7**.
Die anderen Fälle sind zusätzliche Übung, wenn ihr noch Zeit habt.
Bei jedem Fallwechsel beginnt eure Auswahl neu.

**Farben der Merkmalskarten:** positiv = rot, negativ = blau, 0 = weiss.
Alle drei Karten verwenden dieselbe Farbskala. Achtet besonders auf die Zahlen.


In [ ]:
#@title 3. Filterdetektive: Beispiel wählen, Merkmalskarte auswählen und prüfen
def get_pattern_image(pattern_name):
    if pattern_name == 'Senkrechte Kante':
        return EDGE_IMAGE.copy()
    if pattern_name == 'Waagrechte Kante':
        return EDGE_IMAGE.T.copy()
    if pattern_name == 'Umgekehrte senkrechte Kante':
        return 1 - EDGE_IMAGE
    if pattern_name == 'Gleichmässige Fläche':
        return np.ones((6, 6), dtype=int)
    image = np.zeros((6, 6), dtype=int)
    image[3:, 3:] = 1
    return image


PATTERN_NAMES = ['Senkrechte Kante', 'Waagrechte Kante',
                 'Umgekehrte senkrechte Kante', 'Gleichmässige Fläche', 'Ecke']
FILTER_CASES = []
for pattern_name in PATTERN_NAMES:
    for orientation in ['senkrechte', 'waagrechte']:
        FILTER_CASES.append((pattern_name, orientation))


def get_filter_case(case_index):
    pattern_name, orientation = FILTER_CASES[case_index]
    image = get_pattern_image(pattern_name)
    kernel = VERTICAL_FILTER if orientation == 'senkrechte' else HORIZONTAL_FILTER
    other_kernel = HORIZONTAL_FILTER if orientation == 'senkrechte' else VERTICAL_FILTER
    correct_map = apply_filter(image, kernel)
    standard_map = apply_filter(EDGE_IMAGE, VERTICAL_FILTER)
    proposed = [apply_filter(image, other_kernel), -correct_map,
                np.rot90(correct_map), standard_map, standard_map.T,
                -standard_map, np.full((4, 4), 3), np.zeros((4, 4), dtype=int)]
    wrong_maps = []
    for candidate in proposed:
        if (not np.array_equal(candidate, correct_map)
                and not any(np.array_equal(candidate, other) for other in wrong_maps)):
            wrong_maps.append(candidate.copy())
        if len(wrong_maps) == 2:
            break
    correct_index = [1, 0, 2, 0, 1, 2, 1, 0, 2, 1][case_index]
    candidates = list(wrong_maps)
    candidates.insert(correct_index, correct_map.copy())
    return {'pattern': pattern_name, 'orientation': orientation, 'image': image,
            'kernel': kernel, 'correct_map': correct_map,
            'candidates': candidates, 'correct_index': correct_index}


def filter_task_png(case_index):
    data = get_filter_case(case_index)
    fig = plt.figure(figsize=(11, 7))
    layout = fig.add_gridspec(2, 6)
    image_axis = fig.add_subplot(layout[0, 1:3])
    filter_axis = fig.add_subplot(layout[0, 3:5])
    draw_grid(image_axis, data['image'], 'Das Bild: 6 × 6', mode='image', font_size=12)
    draw_grid(filter_axis, data['kernel'], 'Der gesuchte 3×3-Filter',
              low=-1, high=1, font_size=17, highlight=(0, 0, 3, BLUE))
    for index, candidate in enumerate(data['candidates']):
        axis = fig.add_subplot(layout[1, index * 2:index * 2 + 2])
        draw_grid(axis, candidate, f'Merkmalskarte {"ABC"[index]}',
                  low=-3, high=3, font_size=16)
    fig.suptitle(f"{data['pattern']}\nFilter für {data['orientation']} Kanten",
                 fontsize=16, fontweight='bold', color=DARK)
    fig.text(0.5, 0.02, 'Welche Merkmalskarte liefert genau dieser Filter für dieses Bild?',
             ha='center', va='bottom', fontsize=11, color=DARK)
    fig.tight_layout(rect=(0.01, 0.07, 0.99, 0.88))
    return figure_png(fig)


class FilterGame:
    def __init__(self):
        labels = [(f'{index + 1}. {pattern} – Filter für {orientation} Kanten', index)
                  for index, (pattern, orientation) in enumerate(FILTER_CASES)]
        self.case_select = widgets.Dropdown(options=labels, value=0, description='Beispiel:',
                                            style={'description_width': 'initial'},
                                            layout=widgets.Layout(width='100%'))
        self.image = widgets.Image(format='png', value=filter_task_png(0),
                                   layout=widgets.Layout(width='100%', height='auto'))
        self.choice = widgets.RadioButtons(options=[('Karte A', 0), ('Karte B', 1), ('Karte C', 2)],
                                           index=None, description='Meine Wahl:',
                                           style={'description_width': 'initial'},
                                           layout=widgets.Layout(width='100%'))
        self.check_button = widgets.Button(description='Prüfen', icon='check',
                                            layout=widgets.Layout(width='145px', height='36px'))
        self.feedback = widgets.HTML()
        self.ui = panel([
            widgets.HTML('<b>Welche Merkmalskarte passt zum Bild und zum gesuchten Filter?</b>'),
            self.case_select, self.image, self.choice, self.check_button, self.feedback,
        ])
        self.case_select.observe(self.change_case, names='value')
        self.choice.observe(self.reset_feedback, names='value')
        self.check_button.on_click(self.check)

    def reset_feedback(self, change=None):
        self.feedback.value = ''

    def change_case(self, change):
        self.choice.index = None
        self.feedback.value = ''
        self.image.value = filter_task_png(self.case_select.value)

    def check(self, button=None):
        if self.choice.value is None:
            self.feedback.value = feedback_html('Wählt zuerst Karte A, B oder C.')
            return
        data = get_filter_case(self.case_select.value)
        if self.choice.value == data['correct_index']:
            self.feedback.value = feedback_html('Richtig. Eure Karte passt zu diesem Bild und Filter.', 'correct')
        else:
            self.feedback.value = feedback_html(
                'Noch nicht richtig. Vergleicht die Filtergewichte mit den Helligkeitsübergängen und versucht es nochmals.',
                'wrong')

filter_game = FilterGame()
display(filter_game.ui)


**Besprecht:** Welcher Filter reagiert auf welche Kante? Kann eine senkrechte Kante
auch zu einem Zaun oder einer Hauswand gehören?

**Negative Antworten sind kein Rechenfehler.** Bei unserem Filter können sie den
umgekehrten Übergang anzeigen: hell links, dunkel rechts.
Später sehen wir, was eine ReLU-Aktivierung mit negativen Werten macht.

**Wichtig für das CNN:** Diese gut lesbaren Filter sind von uns vorgegeben.
In einem trainierten CNN werden die Filtergewichte beim Training angepasst.
Die gelernten Muster müssen nicht so einfach aussehen oder einen eindeutigen Namen haben.


## 4 · Max-Pooling: weniger Werte · 15 Minuten

Nach der Faltung können Merkmalskarten viele Werte enthalten.
**Pooling fasst kleine Bereiche zusammen.**

Wir verwenden hier **Max-Pooling mit 2×2-Bereichen**, die sich nicht überlappen.

1. Teilt die Karte in vier getrennte Bereiche auf.
2. Sucht in **jedem Bereich** den grössten Wert.
3. Übernehmt diesen Wert an die passende Position der kleineren Karte.

Es werden **nicht die vier grössten Werte der ganzen Karte** gesucht.
Jeder Bereich liefert genau einen Wert.

### 4a · Erst selber vorhersagen

Die Farben markieren zusammengehörende Bereiche.
Welche vier Werte gehören in die kleine Karte?
Tragt eure Vermutung in die vier Eingabefelder unter der Grafik ein und klickt auf **«Prüfen»**.
**4b** bleibt die anschliessende Lösung zum Vergleichen.


In [ ]:
#@title 4a. Pooling-Aufgabe anschauen: vier Bereiche, vier Ergebnisse
pool_input = np.array([
    [1, 3, 0, 2],
    [4, 2, 1, 5],
    [2, 0, 6, 1],
    [1, 3, 4, 2],
])
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
draw_grid(axes[0], pool_input, 'Merkmalskarte: vier 2×2-Bereiche', blocks=True, font_size=19)
draw_grid(axes[1], np.zeros((2, 2), dtype=int), 'Eure Vermutung: ein Wert pro Farbe',
          blocks=True, font_size=23)
for label in axes[1].texts:
    label.set_text('?')
finish_figure(fig, 'Max-Pooling: Sucht pro Bereich den grössten Wert',
              'Die vier Ergebnisfelder behalten die Anordnung der vier Bereiche.')


In [ ]:
#@title 4a. Eure vier Pooling-Werte eingeben und prüfen
class PoolingTask:
    def __init__(self):
        labels = ['Oben links:', 'Oben rechts:', 'Unten links:', 'Unten rechts:']
        self.fields = [widgets.Text(value='', placeholder='Zahl eingeben', description=label,
                                    style={'description_width': 'initial'},
                                    layout=widgets.Layout(width='100%')) for label in labels]
        self.grid = widgets.GridBox(self.fields, layout=widgets.Layout(
            width='100%', max_width='700px', grid_template_columns='repeat(2, minmax(160px, 1fr))',
            grid_gap='10px 20px'))
        self.check_button = widgets.Button(description='Prüfen', icon='check',
                                            layout=widgets.Layout(width='145px', height='36px'))
        self.feedback = widgets.HTML()
        self.ui = panel([
            widgets.HTML('<b>Eure Vermutung für die kleinere Karte</b><br>'
                         'Jeder farbige 2×2-Bereich liefert einen Wert. Die Anordnung bleibt erhalten.'),
            self.grid, self.check_button, self.feedback,
        ])
        for field in self.fields:
            field.observe(self.reset_feedback, names='value')
        self.check_button.on_click(self.check)

    def reset_feedback(self, change=None):
        self.feedback.value = ''

    def check(self, button=None):
        raw_values = [field.value.strip() for field in self.fields]
        if not all(raw_values):
            self.feedback.value = feedback_html('Füllt zuerst alle vier Felder aus.')
            return
        try:
            entered = np.array([int(value) for value in raw_values]).reshape(2, 2)
        except ValueError:
            self.feedback.value = feedback_html('Gebt in jedes Feld eine ganze Zahl ein.')
            return
        if np.array_equal(entered, max_pool(pool_input)):
            self.feedback.value = feedback_html('Richtig. Alle vier Werte sind an der passenden Position.', 'correct')
        else:
            self.feedback.value = feedback_html(
                'Noch nicht richtig. Prüft jeden farbigen 2×2-Bereich einzeln und achtet auf oben, unten, links und rechts.',
                'wrong')

pool_task = PoolingTask()
display(pool_task.ui)


In [ ]:
#@title 4b. Max-Pooling auflösen und die Herkunft jedes Werts prüfen
pooled_result = max_pool(pool_input)
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
draw_grid(axes[0], pool_input, 'Ein eingekreistes Maximum pro Bereich',
          blocks=True, ring_maxima=True, font_size=19)
draw_grid(axes[1], pooled_result, 'Kleinere Merkmalskarte: 2 × 2', blocks=True, font_size=24)
finish_figure(fig, 'Max-Pooling übernimmt lokale Maximalwerte',
              '4 × 4 = 16 Werte werden zu 2 × 2 = 4 Werten. Das ist ein Viertel der Werte.')
print('Oben links: max(1, 3, 4, 2) = 4')
print('Oben rechts: max(0, 2, 1, 5) = 5')
print('Unten links: max(2, 0, 1, 3) = 3')
print('Unten rechts: max(6, 1, 4, 2) = 6')


### 4c · Überraschung: Können wir das Original zurückholen?

Zwei Teams erhalten unterschiedliche Merkmalskarten: **A** und **B**.
Beide wenden dieselbe Max-Pooling-Regel an.

**Vermutet zuerst:** Müssen unterschiedliche Karten auch unterschiedliche
Pooling-Ergebnisse liefern?


In [ ]:
#@title 4c. Zwei verschiedene Karten mit gleichem Pooling-Ergebnis
secret_map_a = np.array([
    [9, 0, 5, 0],
    [0, 0, 0, 0],
    [7, 0, 4, 0],
    [0, 0, 0, 0],
])
secret_map_b = np.array([
    [0, 9, 1, 5],
    [8, 2, 3, 4],
    [1, 2, 0, 1],
    [7, 6, 3, 4],
])
pooled_a = max_pool(secret_map_a)
pooled_b = max_pool(secret_map_b)
fig, axes = plt.subplots(1, 3, figsize=(11.5, 4.5))
draw_grid(axes[0], secret_map_a, 'Karte A', blocks=True, ring_maxima=True, font_size=19)
draw_grid(axes[1], secret_map_b, 'Karte B: andere Werte und Positionen',
          blocks=True, ring_maxima=True, font_size=19)
draw_grid(axes[2], pooled_a, 'Dasselbe Ergebnis für A und B', blocks=True, font_size=24)
finish_figure(fig, 'Nach dem Pooling sind A und B nicht mehr unterscheidbar',
              'Die Farbe zeigt den Bereich. Ein Kreis markiert jeweils einen grössten Wert.')
print('A und B sind verschieden:', not np.array_equal(secret_map_a, secret_map_b))
print('Ihre Pooling-Ergebnisse sind gleich:', np.array_equal(pooled_a, pooled_b))


**Was bleibt erhalten?** Der grösste Zahlenwert jedes 2×2-Bereichs.

**Was geht verloren?** Die übrigen Werte und die genaue Position des Maximums
innerhalb des Bereichs. Aus dem Ergebnis allein können wir die Ausgangskarte nicht
eindeutig rekonstruieren.

Pooling kann starke Antworten mit weniger Werten weitergeben. Es weiss aber nicht,
welche Information für unsere spätere Aufgabe wichtig ist.
**Es kann auch nützliche Details verlieren.**

Eine kleine Verschiebung kann innerhalb eines Bereichs zum gleichen Ergebnis führen.
Daraus folgt nicht, dass ein CNN jede Verschiebung, Drehung oder jeden Ausschnitt
problemlos erkennt.

**Die Max-Pooling-Regel wird vorgegeben. Sie lernt keine Filtergewichte.**


**Labordokumentation (Word) · Experiment 2:**
Haltet euren Kurzbefund zu Max-Pooling und zum Informationsverlust dort fest.


## 5 · Die Bausteine im CNN · 8 Minuten

**CNN** steht für **Convolutional Neural Network**, auf Deutsch etwa
«faltendes neuronales Netzwerk».

Jetzt verbinden wir drei Bausteine:

- **Faltung:** Ein Filter berechnet Antworten auf kleine Bildmuster.
- **ReLU:** Negative Werte werden 0. Positive Werte und 0 bleiben gleich.
  Die Karte behält dabei ihre Anzahl an Zeilen und Spalten.
- **Max-Pooling:** Jeder 2×2-Bereich liefert seinen grössten Wert.

**Vermutet vor dem Ausführen:** Welche Werte verändert ReLU?
Welche Werte stehen danach im Pooling-Ergebnis?


In [ ]:
#@title 5. Ein Bild durch Faltung, ReLU und Max-Pooling verfolgen
pipeline_image = np.tile([0, 0, 1, 1, 0, 0], (6, 1))
pipeline_map = apply_filter(pipeline_image, VERTICAL_FILTER)
activated_map = np.maximum(pipeline_map, 0)
pipeline_result = max_pool(activated_map)
fig, axes = plt.subplots(1, 4, figsize=(12, 4.4))
draw_grid(axes[0], pipeline_image, 'Bild\n6 × 6', mode='image', font_size=12)
draw_grid(axes[1], pipeline_map, 'Faltung\n4 × 4', low=-3, high=3, font_size=15)
draw_grid(axes[2], activated_map, 'ReLU\n4 × 4', low=-3, high=3, font_size=15)
draw_grid(axes[3], pipeline_result, 'Max-Pooling\n2 × 2', low=-3, high=3, font_size=23)
finish_figure(fig, 'Jeder Baustein hat eine andere Aufgabe',
              'ReLU: −3 wird 0. Max-Pooling: pro 2×2-Bereich ein Maximum. '
              'Die Filtergewichte sind hier vorgegeben.')


Ein CNN verwendet mehrere Filter und oft mehrere solcher Verarbeitungsstufen.
Spätere Schichten können einfache Muster zu komplexeren Mustern kombinieren.
Am Ende berechnet eine **Klassifikation** die Zuordnung zu den gelernten Klassen,
zum Beispiel **«Pferd»** oder **«Mensch»**.

Unsere Filterkarte allein trifft diese Entscheidung noch nicht.
Eine Kante kann zu einem Tier, einer Person oder zum Hintergrund gehören.

| Baustein | Was passiert? | Werden beim CNN-Training Gewichte gelernt? |
|---|---|---|
| Faltung | Derselbe Filter wird an vielen Bildstellen angewandt. | **Ja, die Filtergewichte werden beim Training angepasst.** |
| ReLU | Negative Werte werden 0. | Nein, diese Regel ist fest. |
| Max-Pooling | Lokale Maximalwerte werden weitergegeben. | Nein, diese Regel ist fest. |

**Training und Anwenden unterscheiden:** Beim Training werden Gewichte angepasst.
Beim späteren Untersuchen eines neuen Bildes verwendet das trainierte Modell seine
gelernten Gewichte. Es lernt nicht allein dadurch weiter, dass ihr ein neues Bild zeigt.

**Erklärt euch gegenseitig in 30 Sekunden:** Warum ist Faltung etwas anderes als Pooling?


## 6 · Lerncheck ohne Hilfe · 9 Minuten

Bearbeitet die Auswahlaufgaben zuerst ohne Zurückscrollen.
Führt die Zelle mit den neuen Aufgabenrastern und danach die Zelle mit den Antworten aus.
Wählt eure Antworten und klickt bei der jeweiligen Frage auf **«Prüfen»**.

**Die Begründungen in ganzen Sätzen und den Rechenweg haltet ihr in der
Labordokumentation (Word) fest.** Dort erklärt ihr auch das Verschieben des Filters,
den Informationsverlust und die Grenze eines einzelnen Kantenfilters.


In [ ]:
#@title 6. Neue Zahlenbeispiele für den Lerncheck anzeigen
check_patch = np.array([[0, 0, 1], [0, 1, 0], [0, 0, 1]])
check_pool_input = np.array([
    [2, 1, 4, 0],
    [0, 3, 2, 1],
    [1, 0, 2, 1],
    [0, 2, 3, 5],
])
fig, axes = plt.subplots(1, 3, figsize=(11, 4.6))
draw_grid(axes[0], check_patch, 'Aufgabe 1 · Bildausschnitt', mode='image', font_size=19)
draw_grid(axes[1], VERTICAL_FILTER, 'Aufgabe 1 · Filter', low=-1, high=1, font_size=19)
draw_grid(axes[2], check_pool_input, 'Aufgabe 2 · Max-Pooling', blocks=True, font_size=18)
finish_figure(fig, 'Jetzt seid ihr die Bilddetektive',
              'Zuerst selber rechnen. Danach Antworten wählen und «Prüfen» klicken.')


In [ ]:
#@title 6. Eure Antworten zum Lerncheck auswählen und prüfen
def quiz_pool_cards_png():
    maps = [np.array([[3, 4], [2, 3]]), np.array([[3, 4], [2, 5]]),
            np.array([[4, 3], [5, 2]])]
    fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
    for index, candidate in enumerate(maps):
        draw_grid(axes[index], candidate, f'Karte {"ABC"[index]}', blocks=True, font_size=22)
    fig.suptitle('Aufgabe 2 · Welche kleinere Karte entsteht?', fontsize=15,
                 fontweight='bold', color=DARK)
    fig.tight_layout(rect=(0.01, 0.01, 0.99, 0.88))
    return figure_png(fig)


class QuizItem:
    def __init__(self, title, options, correct_value, kind='radio', image=None):
        self.kind = kind
        self.correct_value = correct_value
        if kind == 'multiple':
            self.inputs = [widgets.Checkbox(value=False, description=label, indent=False,
                                            layout=widgets.Layout(width='100%')) for label in options]
            self.control = widgets.VBox(self.inputs)
            for control in self.inputs:
                control.observe(self.reset_feedback, names='value')
        elif kind == 'dropdown':
            self.control = widgets.Dropdown(options=[('Bitte wählen …', None), *options], value=None,
                                            layout=widgets.Layout(width='100%'))
            self.control.observe(self.reset_feedback, names='value')
        else:
            self.control = widgets.RadioButtons(options=options, index=None,
                                                layout=widgets.Layout(width='100%'))
            self.control.observe(self.reset_feedback, names='value')
        self.check_button = widgets.Button(description='Prüfen', icon='check',
                                            layout=widgets.Layout(width='145px', height='34px'))
        self.feedback = widgets.HTML()
        children = [widgets.HTML(f'<b>{escape(title)}</b>')]
        if image is not None:
            children.append(widgets.Image(value=image, format='png',
                                           layout=widgets.Layout(width='100%', height='auto')))
        children += [self.control, self.check_button, self.feedback]
        self.ui = widgets.VBox(children, layout=widgets.Layout(
            width='100%', padding='12px 0', border_bottom='1px solid #cbd5e1'))
        self.check_button.on_click(self.check)

    def reset_feedback(self, change=None):
        self.feedback.value = ''

    def selected_value(self):
        if self.kind == 'multiple':
            return frozenset(index for index, control in enumerate(self.inputs) if control.value)
        return self.control.value

    def check(self, button=None):
        selected = self.selected_value()
        if selected is None or (self.kind == 'multiple' and not selected):
            self.feedback.value = feedback_html('Wählt zuerst eine Antwort.' if self.kind != 'multiple'
                                                else 'Markiert zuerst die Aussagen, die ihr für richtig haltet.')
        elif selected == self.correct_value:
            self.feedback.value = feedback_html('Richtig.', 'correct')
        else:
            self.feedback.value = feedback_html('Noch nicht richtig. Überlegt nochmals und prüft erneut.', 'wrong')


class LearningQuiz:
    def __init__(self):
        self.items = [
            QuizItem('1 · Welche Filterantwort ergibt der neue 3×3-Bildausschnitt?',
                     [('−2', -2), ('0', 0), ('2', 2), ('3', 3)], 2),
            QuizItem('2 · Welche Karte entsteht beim Max-Pooling aus den vier farbigen Bereichen?',
                     [('Karte A', 0), ('Karte B', 1), ('Karte C', 2)], 1, image=quiz_pool_cards_png()),
            QuizItem('3 · Wie verändert sich die Anzahl der Werte beim gezeigten Pooling?',
                     [('16 Werte → 4 Werte', '16_to_4'), ('16 Werte → 8 Werte', '16_to_8'),
                      ('4 Werte → 16 Werte', '4_to_16')], '16_to_4', kind='dropdown'),
            QuizItem('4 · Was macht ReLU aus den Werten −2, 0, 3?',
                     [('−2, 0, 3', 'unchanged'), ('0, 0, 3', 'relu'), ('0, 0, 0', 'zeros')], 'relu'),
            QuizItem('5 · Was passiert bei ReLU mit der Zahl der Zeilen und Spalten?',
                     [('Sie bleibt gleich.', 'same'), ('Sie wird halbiert.', 'half'),
                      ('Sie wird verdoppelt.', 'double')], 'same', kind='dropdown'),
            QuizItem('6 · Welche Aussagen passen? Wählt alle richtigen Aussagen.', [
                'Die Filtergewichte werden beim Training angepasst.',
                'Max-Pooling lernt beim Training eine neue Auswahlregel.',
                'ReLU wendet eine feste Regel an.',
                'Beim Anwenden des trainierten Modells bleiben seine Gewichte unverändert.',
            ], frozenset({0, 2, 3}), kind='multiple'),
        ]
        self.ui = panel([
            widgets.HTML('<b>Eure Antworten zum Lerncheck</b><br>'
                         'Wählt pro Auswahlfrage eine Antwort. Bei Frage 6 sind mehrere Aussagen richtig.'),
            *[item.ui for item in self.items],
        ])

learning_quiz = LearningQuiz()
display(learning_quiz.ui)


**Labordokumentation (Word) · Lerncheck:**
Ergänzt dort eure schriftlichen Erklärungen und den Rechenweg.
Die Auswahlaufgaben zur Selbstkontrolle bearbeitet ihr hier im Colab.


### Das nehmt ihr in die Prüfungsvorbereitung mit

**Faltung:** passende Werte multiplizieren, Produkte addieren, denselben Filter verschieben.
Die Antworten stehen in einer neuen Merkmalskarte.

**ReLU:** negative Werte werden 0. Die räumliche Grösse bleibt gleich.

**Max-Pooling:** pro getrenntem 2×2-Bereich ein Maximum übernehmen.
Weniger Werte bedeuten hier auch Informationsverlust.

**CNN:** Filtergewichte werden beim Training gelernt. Mehrere Schichten verarbeiten
Muster, bevor die Klassifikation die Bildklasse berechnet.

**Noch offen für Teil 2:** Wie gut erkennt ein vorbereitetes CNN echte Fotos von Pferden
und Menschen? Was passiert bei Ausschnitten, ungewöhnlichen Posen oder anderen Hintergründen?


Copyright 2026 [Markus Ineichen](mailto:markus.ineichen1@sluz.ch)

Code licence: [MIT License](https://mit-license.org/)

Text license: [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International](https://creativecommons.org/licenses/by-nc-sa/4.0/)